## Linear Regression Model

In [ ]:
import numpy as np
import pandas as pd


def predict(X, w):
    if isinstance(X, pd.DataFrame) or isinstance(X, pd.Series):
        X_val = X.values
    else:
        X_val = X
    return X_val.dot(w)


def compute_cost(X, y, w):
    m = len(y)
    h = predict(X, w)
    # Ensure y is a column vector for subtraction
    if isinstance(y, pd.Series):
        y_val = y.values.reshape(-1, 1)
    else:
        y_val = y # Assuming y is already a numpy column vector
    cost = (1/(2*m)) * np.sum((h - y_val)**2)
    return cost


def mini_batch_gradient_descent(X, y, w, alpha, epochs, batch_size):
    m = len(y)
    cost_history = []

    for epoch in range(epochs):

        # Process each mini-batch
        for i in range(0, m, batch_size):
            X_batch = X[i:i + batch_size] # DataFrame slice
            y_batch = y[i:i + batch_size] # Series slice

            # Forward pass
            # predict function now handles conversion of X_batch to its values
            h = predict(X_batch, w)
            # Gradient for d features
            gradient = (1 / len(y_batch)) * X_batch.values.T.dot(h - y_batch.values.reshape(-1, 1))
            # Parameter update
            w = w - alpha * gradient

        cost_history.append(compute_cost(X, y, w))

    return w, cost_history

## Loading and Preparing Dataset

In [ ]:
data = pd.read_csv('/content/advertising.csv')
train = data.sample(frac=0.8, random_state=0)
test = data.drop(train.index) # remaining 20%

X_ori = train.iloc[:, :-1] # DataFrame of original 8 features
X_ori_mean = X_ori.mean(axis=0)
X_ori_std  = X_ori.std(axis=0)
X_norm = (X_ori - X_ori_mean) / X_ori_std
y_target = train.iloc[:, -1]    # Series of target values

# Add a bias (intercept) term to X_features_original
# Create a DataFrame for the bias term (a column of ones)
bias_column = pd.DataFrame(np.ones((X_norm.shape[0], 1)),
                           index=X_norm.index,
                           columns=['bias'])

X = pd.concat([bias_column, X_norm], axis=1)
# Assign y for gradient descent
y = y_target

## Training the Model

In [ ]:
# Number of features including the bias term
num_features_with_bias = X.shape[1]
# Initial weights (zeros) - now correctly sized for (d+1) features
w_init = np.zeros((num_features_with_bias, 1))

# Hyperparameters
alpha = 0.001
epochs = 200
batch_size = 16

w_final, cost_history = mini_batch_gradient_descent(X, y, w_init, alpha, epochs, batch_size)

print("Final learned weights:")
print(w_final)
print("Final cost:", cost_history[-1])

## Testing the Model

In [ ]:
X_test_ori = test.iloc[:, :-1]
y_test = test.iloc[:, -1]

# Normalize test features using TRAIN mean & std
X_test_norm = (X_test_ori - X_ori_mean) / X_ori_std

# Add bias column
bias_test = pd.DataFrame(np.ones((X_test_norm.shape[0], 1)),
                         index=X_test_norm.index,
                         columns=['bias'])

X_test = pd.concat([bias_test, X_test_norm], axis=1)

# Compute predictions
y_pred_test = predict(X_test, w_final)

# Compute test MSE (same as cost)
test_cost = compute_cost(X_test, y_test, w_final)

print("Test cost (MSE):", test_cost)

# Optional: print first 10 predictions vs actual
results = pd.DataFrame({
    'Actual': y_test.values.flatten(),
    'Predicted': y_pred_test.flatten()
})est cost (MSE): 3.7805979554523916



print("\nSample Test Predictions:")
print(results.head(10))

Test cost (MSE): 3.7805979554523916

Sample Test Predictions:
   Actual  Predicted
0    15.6  13.620504
1    17.5  15.647947
2    17.0  16.665309
3    10.5   8.915969
4    11.9  11.012402
5    13.2   8.779374
6    25.4  20.159453
7    21.5  18.138058
8    23.2  18.856135
9    23.8  18.428498
